In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from quickrules.data_handling import test_save
from quickrules.quickrules import QuickRules
from quickrules.relations import StatisticalRelation, RelationFactory, MinMaxRelation
from quickrules.fuzzy_operators import MinTNorm, KleeneDienesImplicator
from quickrules.owaquickrules import OWAQuickRules
from quickrules.weights import ExponentialWeights, LinearWeights, TruncatedWeights
from pathlib import Path
import multiprocessing as mp

In [3]:
data_sets = [
    'australian',
     'cleveland',
     'glass',
     'heart',
     'ionosphere',
     'pima',
     'wine',
    'bupa',
    'saheart',
    'wisconsin',
    # 'balance',  # same as monk
    'crx',
    'ecoli',
    'haberman',
    # 'monk',  # problems with list indices again
    'mammographic'
             ]

extra_data_sets = [  # commented out any above 2000 samples
    'automobile',
    # 'banana',
    'bands',
    'contraceptive',
    'dermatology',
    'german',
    # 'marketing',
    'movement',
    # 'page-blocks',
    # 'phoneme',
    # 'ring',
    # 'satimage',
    # 'segment',
    'sonar',
    # 'spambase',
    'spectfheart',
    # 'texture',
    # 'titanic',
    # 'thyroid',
    # 'twonorm',
    'vehicle',
    'vowel',
    # 'wdbc',
    'winequality-red',
    # 'winequality-white',
    'yeast'
]

exclude = [
    'thyroid',  # too big
    'wdbc',  # almost done
]

In [4]:
relation_factory = RelationFactory(MinMaxRelation, MinTNorm())

models = {
    'qr': ('close-min-max-combo-results',
           QuickRules(MinTNorm(), KleeneDienesImplicator(), relation_factory, prune=False)),
    'lin-owa': ('rules-lin-owa-qr-combo-minmax-results',
                OWAQuickRules(MinTNorm(), KleeneDienesImplicator(), relation_factory, LinearWeights(), prune=False)),
    'trun-lin-owa': ('rules-trun-lin-owa-qr-combo-minmax-results',
                     OWAQuickRules(MinTNorm(), KleeneDienesImplicator(), relation_factory, TruncatedWeights(weights=LinearWeights(), cut_off=10), prune=False)),
    'trun-exp-owa': ('trun-exp-owa-qr-combo-minmax-results',
                     OWAQuickRules(MinTNorm(), KleeneDienesImplicator(), relation_factory, TruncatedWeights(weights=ExponentialWeights(), cut_off=10), prune=False)),
    'prune-qr': ('prune-qr-combo-minmax-results',
                 QuickRules(MinTNorm(), KleeneDienesImplicator(), relation_factory, prune=True)),
    'prune-lin-owa': ('prune-lin-owa-qr-combo-minmax-results',
                      OWAQuickRules(MinTNorm(), KleeneDienesImplicator(), relation_factory, LinearWeights(), prune=True)),
    'prune-trun-lin-owa': ('prune-trun-lin-owa-qr-combo-min-max-results',
                           OWAQuickRules(MinTNorm(), KleeneDienesImplicator(), relation_factory, TruncatedWeights(weights=LinearWeights(), cut_off=10), prune=True)),
    'prune-trun-exp-owa': ('prune-trun-exp-owa-qr-combo-min-max-results',
                           OWAQuickRules(MinTNorm(), KleeneDienesImplicator(), relation_factory, TruncatedWeights(weights=ExponentialWeights(), cut_off=10), prune=True))
}

In [5]:
data_path = Path.cwd() / 'keel-data'
processes = []

for name, params in models.items():
    processes.append(mp.Process(
        target=test_save,
        kwargs= {
            'model': params[1],
            'datasets_folder': data_path,
            'results_folder': Path.cwd() /params[0],
            'get_rules': True,
            'print_info': True,
            'include': extra_data_sets,
            'verbose' : True
        },
        name=name
    ))

for process in processes:
    process.start()

In [8]:
for process in processes:
    print("Process {name} is {status}.".format(
        name=process.name,
        status="alive" if process.is_alive() else "dead"
))

Process qr is alive.
Process lin-owa is alive.
Process trun-lin-owa is alive.
Process trun-exp-owa is alive.
Process prune-qr is alive.
Process prune-lin-owa is alive.
Process prune-trun-lin-owa is alive.
Process prune-trun-exp-owa is alive.


In [13]:
# Only run when wanting to abort the processes
for process in processes:
    process.terminate()